# Select from annotations table

**Author:** Juliana Thomasia C. Marcos\
**Input:** Databricks annotations table (`TABLE_NAME`), associated images and a new or existing Roboflow project information\
**Output:** Annotations and images uploaded to a Roboflow project for model training

## Overview

This notebook selects and uploads a subset of annotations and associated images from an existing Databricks annotations table to a Roboflow project for model development and training. It supports the creation or retrieval of a Roboflow project and performs data subsampling based on some attributes to promote diversity in the training dataset. The selected images and their corresponding annotations are temporarily stored in a designated staging folder before further processing or upload.

The notebook supports optional image deduplication through the GIST pipeline. When deduplication is enabled, the selected images must first be processed through GIST before the resulting images and their corresponding annotations can be uploaded to Roboflow. Otherwise, the annotated images are uploaded directly from the staging folder. The resulting dataset can then be used to train models for detecting or segmenting specific crop traits, enabling subsequent inference on new images. Following a successful upload, temporary files in the staging and GIST folders are removed as applicable.

# Packages import and installation

In [0]:
%pip install ..

In [0]:
dbutils.library.restartPython()

In [0]:
import shutil
import os
import pandas as pd

from roboflow import Roboflow
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from matplotlib import pyplot as plt
from datetime import datetime

from ag_vision.data_io import databricks_io as dbio
from ag_vision.data_io import roboflow_io as rio
from ag_vision.data_io import annotation_io as aio
from ag_vision.constants import paths

# Variables set up

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
rf_workspace = "cgiar-workspace"
api_key = dbutils.secrets.get("enablement-package", "roboflow_api_key")
roboflow_folder_group = dbutils.secrets.get("enablement-package", "roboflow_folder_group")

In [0]:
table_name = dbutils.widgets.get("table_name")
TABLE_NAME = f"{catalog_name}.production.{table_name}"
date = str(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
batch_name=f'batch {date}'
dtype_folder_name = "annotations"

In [0]:
rf_project_name = dbutils.widgets.get("rf_project_name")
annotation_type = dbutils.widgets.get("annotation_type")
annotation_group = dbutils.widgets.get("annotation_group")
staging_folder = dbutils.widgets.get("staging_folder")
n_samples = int(dbutils.widgets.get("n_samples"))
deduplication = dbutils.widgets.get("deduplication") == "True"
table_filtering_keyword = dbutils.widgets.get("table_filtering_keyword")
table_filtering_date = dbutils.widgets.get("table_filtering_date")

In [0]:
gist_url = f"https://dbc-dp-3718194974443840.cloud.databricks.com/driver-proxy/o/3718194974443840/0824-114636-ygb68kcg/8080/"
gist_project_path, roboflow_project_id =  "", ""
staging_project_base_path = f"/Volumes/{catalog_name}/production/data/{staging_folder}"

# Roboflow project creation or retrieval

In [0]:
if annotation_group and api_key and rf_workspace:
  rf_project_id = rio.create_roboflow_project_restapi(project_name=rf_project_name,
                                                      task_type=annotation_type,
                                                      annotation_group=annotation_group,
                                                      api_key=api_key,
                                                      workspace_name=rf_workspace,
                                                      folder_group=roboflow_folder_group
                                                      )

In [0]:
if rf_project_id:
      rf_project_id = rf_project_id.split("/")[1]
      rf = Roboflow(api_key=api_key)
      rf_project = rf.workspace(rf_workspace).project(rf_project_id)

# Images table query and subsampling

In [0]:
annos_df = (spark.table(TABLE_NAME)
    .select('image_path', 'annotation_type', 'download_date', 'annotation_path', 'project')
    .filter((F.col('image_path').like(f'%{table_filtering_keyword}%')) &
            (F.col('download_date') >= table_filtering_date))
    .toPandas())

annos_df.head()

In [0]:
df_list = []
sample_size = 10
final_df = pd.DataFrame()

for idx, df in annos_df.groupby(['download_date']):
    if len(df) < sample_size:
        df_list.append(df)
    else:
        df_list.append(df.sample(sample_size))

if len(df_list):
    final_df = pd.concat(df_list)

In [0]:
if len(final_df) > n_samples:
    final_df = final_df.sample(n_samples)
len(final_df)

# Images deduplication and/or upload to Roboflow


In [0]:
try:    
    aio.add_annotations_to_batch(annotations_df=final_df,
                                 project_path=staging_project_base_path,
                                 annotation_type=annotation_type,
                                 task_name=rf_project_id,
                                 batch_name=batch_name,
                                 date=date)
    
    staging_project_path = f"{staging_project_base_path}/{dtype_folder_name}/{annotation_type}/{rf_project_id}/{batch_name}/images"

    paths.show_volume_folder_link(path=staging_project_path,
                                  url_name="Annotations batch",
                                  prefix=f"Batch successfully created and available at: ",
                                  final_index=True)
except Exception as e:
    print("Something wrong happens during the batch creation and upload to the temporary folder.")

In [0]:
if deduplication:
    print("The selected images can be processed through the GIST pipeline available at: ", gist_url)
    gist_project_base_path = f"/Volumes/{catalog_name}/production/data/gist"
    
    gist_project_path = f"{gist_project_base_path}/{dtype_folder_name}/{annotation_type}/{rf_project_id}/{batch_name}/images"
    if not os.path.exists(gist_project_path):
        os.makedirs(gist_project_path)

    paths.show_volume_folder_link(path=gist_project_path,
                                  url_name="Annotations batch of the selected images",
                                  prefix=f"In order to proceed to the selected images upload to Roboflow, ensure that they are already available at: ",
                                  final_index=True)

    # Annotations copy to the GIST folder
    staging_anno_path = f"{staging_project_base_path}/{dtype_folder_name}/{annotation_type}/{rf_project_id}/{batch_name}/{batch_name.split(' ')[1]}"
    gist_annotations_path = f"{gist_project_base_path}/{dtype_folder_name}/{annotation_type}/{rf_project_id}/{batch_name}/{batch_name.split(' ')[1]}"
    shutil.copytree(src=staging_anno_path, dst=gist_annotations_path, dirs_exist_ok=True)

In [0]:
roboflow_url = f"https://app.roboflow.com/{rf_workspace}/{rf_project_id}/annotate"
still_processing = False

if deduplication:
    if os.path.exists(gist_project_path) and os.listdir(gist_project_path):
        # Roboflow will check for MD5-type (to check) base duplicate images and skip them on upload.
        # That is why we have some missing images in RF.
        rio.upload_annotation_batch_to_roboflow(rf_project=rf_project,
                                                project_path=gist_project_base_path,
                                                annotation_type=annotation_type,
                                                task_name=rf_project_id,
                                                batch_name=batch_name,
                                                download_date=date,
                                                split='train',
                                                tmp_copy = True)
        print(f"\n The uploaded bacth, {batch_name} should now be available in Roboflow at: {roboflow_url}")
    else:
        still_processing = True
        print("\n Results from the GIST pipeline are not available yet...")
else:
        rio.upload_annotation_batch_to_roboflow(rf_project=rf_project,
                                                project_path=staging_project_base_path,
                                                annotation_type=annotation_type,
                                                task_name=rf_project_id,
                                                batch_name=batch_name,
                                                download_date=date,
                                                split='train',
                                                tmp_copy = True)
        print(f"\n The uploaded bacth, {batch_name} should now be available in Roboflow at:\n {roboflow_url}")

In [0]:
if os.path.exists(staging_project_path) and not still_processing:
    staging_project_path = f"{staging_project_base_path}/annotations/{annotation_type}/{rf_project_id}"
    shutil.rmtree(staging_project_path)
    print(f"The temporary files at {staging_project_path} were successfully deleted.")

In [0]:
if deduplication and os.path.exists(gist_project_path) and not still_processing:
    gist_project_path = f"{gist_project_base_path}/annotations/{annotation_type}/{rf_project_id}"
    shutil.rmtree(gist_project_path)
    print(f"The temporary files at {gist_project_path} were successfully deleted.")